# RG3 RH 및 데이터 품질 점검
팀 Confluence의 가설을 원본 데이터로 확인. 신규 분류 모델 학습 없이 기존 Shot/RH 데이터 동일성, 센서 정의, 직전 이력의 0 모드 후보를 점검한다.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
root = Path.cwd()
if not (root/'data').exists(): root=root.parent
old=json.loads((root/'notebooks/10_original_defect_report_revision.ipynb').read_text(encoding='utf-8'))
prep=''.join(next(c for c in old['cells'] if c['cell_type']=='code')['source'])
exec(prep.split('for feature in features:')[0])
auditdir=root/'data/audit'
auditdir.mkdir(exist_ok=True)
rg=labels.loc[labels['product'].eq('RG3')].copy()
rh=rg.loc[rg.side.eq('RH')].copy()
lh=rg.loc[rg.side.eq('LH')].copy()
assert not rh.duplicated(keys).any()
rh.to_csv(auditdir/'rg3_rh_labeled_with_timestamp.csv',index=False,encoding='utf-8-sig')
lh.to_csv(auditdir/'rg3_lh_labeled_with_timestamp.csv',index=False,encoding='utf-8-sig')
current=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
current=current.loc[current['product'].eq('RG3')]
matched=rh.merge(current,on=keys,suffixes=('_rh','_shot'),validate='one_to_one')
assert len(matched)==591
maxdiff=max((matched[f+'_rh']-matched[f+'_shot']).abs().max() for f in features)
assert maxdiff<1e-8
assert matched['PassOrFail_rh'].equals(matched['PassOrFail_shot'])
gas=rh.Reason.str.contains('가스',na=False)
normal=rh.PassOrFail.eq(0)
target=rh.loc[gas|normal].copy()
target['is_gas']=target.Reason.str.contains('가스',na=False)
temps=['Barrel_Temperature_'+str(i) for i in range(1,7)]
valid=observed[['Barrel_Temperature_1','Max_Screw_RPM','Average_Screw_RPM']].notna().all(axis=1)
observed['zero_mode_candidate']=(observed.Barrel_Temperature_1.eq(0) & observed.Max_Screw_RPM.eq(0) & observed.Average_Screw_RPM.eq(0)).astype(float).where(valid)
zero_summary=observed.groupby('product').agg(observed_shots=('TimeStamp','size'),zero_candidates=('zero_mode_candidate','sum'),known_mode=('zero_mode_candidate','count'))
zero_summary.to_csv(tables/'rg3_audit_zero_mode_overall.csv',encoding='utf-8-sig')
coverage=[]
for w in [10,100,300]:
    col='past'+str(w)+'_zero_count'
    observed[col]=observed.groupby('history_segment')['zero_mode_candidate'].transform(lambda s:s.shift(1).rolling(w,min_periods=1).sum())
    observed['known_'+str(w)]=observed.groupby('history_segment')['zero_mode_candidate'].transform(lambda s:s.shift(1).rolling(w,min_periods=1).count())
    check=target.merge(observed[keys+[col,'known_'+str(w)]],on=keys,validate='one_to_one')
    for name,part in check.groupby('is_gas'):
        coverage.append(dict(window=w,group='gas' if name else 'normal',shots=len(part),with_any_zero=int(part[col].gt(0).sum()),full_known_history=int(part['known_'+str(w)].eq(w).sum()),max_zero_count=part[col].max()))
coverage=pd.DataFrame(coverage)
coverage.to_csv(tables/'rg3_audit_zero_mode_history.csv',index=False,encoding='utf-8-sig')
checks=pd.DataFrame([
dict(check='Max_Back_Pressure < Average_Back_Pressure',count=int((target.Max_Back_Pressure<target.Average_Back_Pressure).sum()),total=len(target)),
dict(check='Max_Screw_RPM < Average_Screw_RPM',count=int((target.Max_Screw_RPM<target.Average_Screw_RPM).sum()),total=len(target))])
checks.to_csv(tables/'rg3_audit_sensor_definition_flags.csv',index=False,encoding='utf-8-sig')
unique=target[features].agg(['nunique','min','max']).T
unique.to_csv(tables/'rg3_audit_sensor_ranges.csv',encoding='utf-8-sig')
normal_shots=current.loc[current.PassOrFail.eq(0)].copy()
corr=normal_shots[features].corr(method='spearman')
corr.to_csv(tables/'rg3_audit_normal_spearman.csv',encoding='utf-8-sig')
relation=pd.DataFrame([dict(a=a,b=b,spearman=corr.loc[a,b]) for a,b in [('Max_Injection_Speed','Max_Injection_Pressure'),('Mold_Temperature_3','Mold_Temperature_4')]])
for minutes in [15,30,60]:
    g=normal_shots.groupby('EQUIP_CD')['TimeStamp'].diff().dt.total_seconds()
    # Labeled observation blocks only; not machine Run identifiers.
    print('labeled normal observation blocks',minutes,int((g.isna()|g.gt(minutes*60)).sum()))
sources=[
('태인 공정 및 변수 분석','https://happysally19.atlassian.net/wiki/spaces/~712020d9e86c720dff4d5485f983212f68575f/pages/17989633'),
('지우 10/2 분석','https://happysally19.atlassian.net/wiki/spaces/~712020d9e86c720dff4d5485f983212f68575f/pages/17072129'),
('1차·2차 연결','https://happysally19.atlassian.net/wiki/spaces/~712020d9e86c720dff4d5485f983212f68575f/pages/18022418')]
lines=['# RG3 RH 데이터·센서·이력 점검','',
'## 결론','',
f'- RH 591개: 정상 {int(normal.sum())}, 가스 {int(gas.sum())}, 기타 불량 {int((~normal&~gas).sum())}.',
f'- 기존 Shot 데이터와 RH 데이터의 24개 센서 최대 차이: {maxdiff:g}. 라벨도 전부 일치. RH 가스 전용 분석은 정상+가스 583개로 기존 실험과 같은 대상이다.',
'- 따라서 RH만 따로 저장한 것으로 기존 가스 모델의 성능이 좋아지지는 않는다. LH/RH를 합쳐 새 실험을 하지는 않았다.',
'- 기존 기본 모델 결과 TP=1, FP=6, FN=16, TN=560은 이전 실험 값이며 이번에 새 모델을 학습한 결과가 아니다. 이번에는 데이터 동일성을 검증했다.',
'','## 0값 이력 점검','',
zero_summary.to_string(),'',
'0 모드 후보 = 배럴 1 및 최대·평균 RPM이 동시에 0. 나머지 배럴과 호퍼는 0이 아니므로 모든 온도가 0인 상태로 해석하지 않는다. 가동 상태 정답은 없으므로 확정 정지로 부르지 않는다. 중복/충돌을 정리한 원본 전체 이력이며 대회 2차 데이터 범위와 다르다.',
'',coverage.to_string(index=False),'',
'직전 창만 사용하고 현재 Shot은 제외. 전체 창이 확보되지 않은 경우를 따로 표시했다. 0 후보가 하나라도 있는 창과 결측/부족 이력은 별개다.',
'','## 센서 정의 점검','',checks.to_string(index=False),'',
'두 컬럼이 같은 단위·구간의 진짜 최대/평균이라면 성립하기 어려운 순서다. 단위·정의가 다를 수도 있어 자동 수정/스케일 보정하지 않았다. Average_Screw_RPM 및 Max_Back_Pressure를 조합 후보에서 제외하고 정의 확인이 우선이다.',
'','## 변수 관계','',relation.to_string(index=False),'',
'정상 RG3 관측 상관관계이며 원인으로 해석하지 않는다. 배럴 온도는 구간별로 유지하고, 평균으로 먼저 압축하지 않는다. 시간 변수는 고유값 수가 매우 적으면 우선순위를 낮춘다.',
'','## 팀 문서와 맞춰 해석할 사항','',
'- 원본 CN7에는 10/27 초기 불량이 포함되지만 대회 2차 CN7에는 없다. RG3 범위는 동일하다.',
'- 라벨/비라벨 중복은 준지도 학습 때 제거해야 한다. 현 과거 이력 구성은 설비·제품·Timestamp 기준 중복을 정리하며 정답 라벨을 이력 입력으로 쓰지 않는다.',
'- 양쪽 센서가 같은 문제는 센서만으로 LH/RH 불량 위치를 구별하기 어렵다는 뜻이다. RH 전용 또는 Shot 불량 탐지가 원리적으로 불가능하다는 뜻은 아니다.',
'- 팀 문서의 F1 상한 .667은 동일 센서 입력으로 양쪽 행에 동일 판정을 할 때의 특정 조건이다. 현재 RH 전용 583개에 그대로 적용하면 안 된다.',
'- RG3 차이가 통계적으로 유의하지 않았다는 결과는 현 표본·방법에서 근거를 못 찾았다는 뜻이다. 신호가 절대 없다는 증명은 아니다.',
'- RH 가스가 배기구 막힘 때문이라는 설명은 직접 배기·보전 기록 확인 전까지 원인 가설로 둔다.',
'- 현재 30분 간격은 실제 Run이 아닌 관측 구간/이력 초기화 규칙이다. 정지·금형교체 기록 없이 진짜 Run이라고 확정하지 않는다.',
'','## 다음 제한된 실험','',
'압력~속도 정상 관계의 이탈 + 배럴 각 구간의 자기 기준 이탈을 소수만 조합하되, 기존 관계 실험과 중복되는 후보는 먼저 확인한다. 후보 선정과 임계값을 테스트 구간을 보며 조절하지 않는다. 같은 RF·같은 홀드아웃 구간에서 TP/FP/FN/TN을 비교한다. 앞선 반복 실험을 보고 골라온 같은 구간의 개선은 독립 검증 결과가 아니므로 보류 구간/새 데이터 확인이 필요하다.',
'','## 확인한 출처','']
lines += ['- ['+title+']('+url+')' for title,url in sources]
lines += ['', 'Git: 원격 브랜치 hayeon/main/taein/jiwoo 존재는 git ls-remote로 확인. 로컬 origin/taein의 연결 노트북 코드는 확인했지만 최신 원격 파일은 API 404 및 인증 미설정으로 읽지 못했다. 로컬 사본을 최신이라고 주장하지 않는다. 커밋·푸시·fetch하지 않았다.']
(root/'reports/rg3_rh_data_quality_audit.md').write_text('\n'.join(lines),encoding='utf-8')
print('RH counts',len(rh),int(normal.sum()),int(gas.sum()),'sensor max difference',maxdiff)
print(zero_summary.to_string())
print(coverage.to_string(index=False))
print(checks.to_string(index=False))
print(relation.to_string(index=False))
print(unique.loc[['Injection_Time','Filling_Time','Max_Screw_RPM']].to_string())


labeled normal observation blocks 15 6
labeled normal observation blocks 30 5
labeled normal observation blocks 60 5
RH counts 591 566 17 sensor max difference 0.0
         observed_shots  zero_candidates  known_mode
product                                             
CN7               44502          24294.0       44453
RG3               66339          29504.0       66233
 window  group  shots  with_any_zero  full_known_history  max_zero_count
     10 normal    566              0                 557             0.0
     10    gas     17              0                  16             0.0
    100 normal    566              0                 483             0.0
    100    gas     17              0                  14             0.0
    300 normal    566              0                 326             0.0
    300    gas     17              0                  10             0.0
                                    check  count  total
Max_Back_Pressure < Average_Back_Pressure    583    583
 